In [1]:
import sys
import torch
import os
# sys.path.append('/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/dpvo')
import dpvo
import collections
import torch
import torch.nn as nn
import onnx
import onnxruntime as ort
import numpy as np
import pandas as pd
import random
verbose = False

# Toggle this to force repeatable test iterations (same RNG reset before each run).
DETERMINISTIC_TEST_MODE = False
seed = 42

def set_all_seeds(seed_value, deterministic=False):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        try:
            torch.use_deterministic_algorithms(True)
        except Exception as exc:
            print(f"Warning: deterministic algorithms not fully enforced: {exc}")

set_all_seeds(seed, deterministic=DETERMINISTIC_TEST_MODE)
print(f"Deterministic test mode: {DETERMINISTIC_TEST_MODE}")
torch.set_printoptions(precision=10, sci_mode=False, linewidth=200)


Deterministic test mode: False


In [2]:
# load onnx models
so = ort.SessionOptions()
so.log_severity_level = 2  # 0 = verbose

onnx_dir = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/andy/onnx'
print(f'Loading onnx update modular model')
module_paths = {'patchify': os.path.join(onnx_dir, "patchify.onnx"),
                'update': os.path.join(onnx_dir, "update.onnx"),
                'corr': os.path.join(onnx_dir, "corr.onnx"),
                'norm': os.path.join(onnx_dir, "norm.onnx"),
                'c1': os.path.join(onnx_dir, "c1.onnx"),
                'c2': os.path.join(onnx_dir, "c2.onnx"),
                'agg_kk': os.path.join(onnx_dir, "agg_kk.onnx"),
                'agg_ij': os.path.join(onnx_dir, "agg_ij.onnx"),
                'gru': os.path.join(onnx_dir, "gru.onnx"),
                'w': os.path.join(onnx_dir, "w.onnx"),
                'd': os.path.join(onnx_dir, "d.onnx")}
module_sessions = {}
for name, module_path in module_paths.items():
    if not os.path.isfile(module_path):
        raise FileNotFoundError(f"ONNX encoder file not found in {onnx_dir} for {name}. Run andy/onnx_conversion.ipynb first.")
    onnx_dir_str = os.path.normpath(str(onnx_dir))
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
    # providers = ["CUDAExecutionProvider"]
    module_sessions[name] = ort.InferenceSession(module_path, sess_options=so, providers=providers)
    if verbose:
        print("=== inputs ===")
        for i in module_sessions[name].get_inputs():
            print(i.name, i.shape, i.type)
        print("=== outputs ===")
        for o in module_sessions[name].get_outputs():
            print(o.name, o.shape, o.type)
    if verbose: print(f'Onnx {name} module loaded: {module_sessions[name]}')

Loading onnx update modular model


2026-04-07 17:08:37.951378120 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 1 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-04-07 17:08:37.993323966 [W:onnxruntime:, transformer_memcpy.cc:74 ApplyImpl] 4 Memcpy nodes are added to the graph main_graph for CUDAExecutionProvider. It might have negative impact on performance (including unable to run CUDA graph). Set session_options.log_severity_level=1 to see the detail logs before this message.
2026-04-07 17:08:38.001373534 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to be correct if indices are not duplicated.
2026-04-07 17:08:38.001387522 [W:onnxruntime:Default, scatter_nd.h:51 ScatterNDWithAtomicReduction] ScatterND with reduction=='none' only guarantees to 

In [3]:
# load pytorch models
project_root = '/home/campus.ncl.ac.uk/c4071391/Projects/DPVO/'

from dpvo.net import VONet

# --- 1. Load the DPVO PyTorch Model ---
# Weights path: project root dpvo.pth or andy/dpvo.pth
pth_model_path = os.path.join(project_root, "dpvo.pth")
if not os.path.isfile(pth_model_path):
    pth_model_path = os.path.join(project_root, "andy", "dpvo.pth")
assert os.path.isfile(pth_model_path), f"Checkpoint not found: {pth_model_path}"

export_device = torch.device("cuda")

model = VONet()

ckpt = torch.load(pth_model_path, map_location="cuda", weights_only=True)
state_dict = ckpt if isinstance(ckpt, dict) and "state_dict" not in ckpt else ckpt.get("state_dict", ckpt)

# Strip DataParallel prefix; drop update.lmbda if present (DPVO load_weights does this)
new_state_dict = collections.OrderedDict()
for k, v in state_dict.items():
    if "update.lmbda" in k:
        print(k)
        continue
    new_state_dict[k.replace("module.", "")] = v
    print(k)

missing, unexpected = model.load_state_dict(new_state_dict, strict=False)
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval().to(export_device)



module.patchify.fnet.conv1.weight
module.patchify.fnet.conv1.bias
module.patchify.fnet.layer1.0.conv1.weight
module.patchify.fnet.layer1.0.conv1.bias
module.patchify.fnet.layer1.0.conv2.weight
module.patchify.fnet.layer1.0.conv2.bias
module.patchify.fnet.layer1.1.conv1.weight
module.patchify.fnet.layer1.1.conv1.bias
module.patchify.fnet.layer1.1.conv2.weight
module.patchify.fnet.layer1.1.conv2.bias
module.patchify.fnet.layer2.0.conv1.weight
module.patchify.fnet.layer2.0.conv1.bias
module.patchify.fnet.layer2.0.conv2.weight
module.patchify.fnet.layer2.0.conv2.bias
module.patchify.fnet.layer2.0.downsample.0.weight
module.patchify.fnet.layer2.0.downsample.0.bias
module.patchify.fnet.layer2.1.conv1.weight
module.patchify.fnet.layer2.1.conv1.bias
module.patchify.fnet.layer2.1.conv2.weight
module.patchify.fnet.layer2.1.conv2.bias
module.patchify.fnet.conv2.weight
module.patchify.fnet.conv2.bias
module.patchify.inet.conv1.weight
module.patchify.inet.conv1.bias
module.patchify.inet.layer1.0.co

/home/campus.ncl.ac.uk/c4071391/miniconda3/envs/dpvo/lib/python3.9/site-packages/dpvo/net.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


VONet(
  (patchify): Patchifier(
    (fnet): BasicEncoder4(
      (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (conv1): Conv2d(3, 32, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (relu1): ReLU(inplace=True)
      (layer1): Sequential(
        (0): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          (norm1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
          (norm2): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        )
        (1): ResidualBlock(
          (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (relu): ReLU(inplace=True)
          

In [4]:
# load dummy_inputs:
INPUT_PAYLOAD_PATH = os.path.join('onnx', 'input_payload_0.pth')

device = 'cuda'

payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)
net = payload['net_in'].float()
ctx = payload['inp'].float()
corr = payload['corr'].float()
ii = payload['ii'].long()
jj = payload['jj'].long()
kk = payload['kk'].long()
E_real = int(net.shape[1])

B, E_real, D = net.shape
_, _, Cc = corr.shape


/tmp/ipykernel_1261452/3930280347.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  payload = torch.load(INPUT_PAYLOAD_PATH, map_location=device)


In [5]:
# helper methods
def bind_torch_inputs(io_binding, inputs: dict, device="cuda", device_id=0):
    for name, tensor in inputs.items():
        if tensor is None:
            continue

        assert tensor.is_cuda, f"{name} must be on GPU for zero-copy"

        io_binding.bind_input(
            name=name,
            device_type=device,
            device_id=device_id,
            element_type=(
                np.float32 if tensor.dtype in (torch.float32, torch.float16)
                else np.int64
            ),
            shape=tuple(tensor.shape),
            buffer_ptr=tensor.data_ptr(),
        )

comparison_rows = []


def record_comparison_summary(model: str, max_abs_diff: float, allclose: bool) -> None:
    comparison_rows.append(
        {"model": model, "max_abs_diff": max_abs_diff, "allclose": allclose}
    )


def bind_torch_output(io_binding, name, tensor, device="cuda", device_id=0):
    io_binding.bind_output(
        name=name,
        device_type=device,
        device_id=device_id,
        element_type=(
            np.float32 if tensor.dtype in (torch.float32, torch.float16) else np.int64
        ),
        shape=tuple(tensor.shape),
        buffer_ptr=tensor.data_ptr(),
    )


def assert_close(name, a, b, atol=1e-8, rtol=1e-5):
    d = (a - b).abs().max().item()
    ok = torch.allclose(a, b, atol=atol, rtol=rtol)
    print(f"{name}: max_abs_diff={d:.3e}  allclose={ok}")
    return d, ok
    # assert ok, f"{name} mismatch"


In [6]:
#--------------------patchify----------------------
comparison_rows.clear()
import glob
import cv2
import os.path as osp

# Match andy/onnx_conversion.ipynb: normalized images [B,N,3,H,W], patches_per_image=96
BATCH, NUM_FRAMES, C, H, W = 1, 1, 3, 480, 640
PATCHES_PER_IMAGE = 96
STRIDE = 1
fx, fy, cx, cy = [320, 320, 320, 240]
imagedir = '/mnt/data/datasets/agricultural/tartanair/tartanair_mono_track/ME005/'

# images_raw = (torch.rand(BATCH, NUM_FRAMES, C, H, W, device="cuda") * 255.0).to(torch.float32)
# from evaluate_tartan.py
def video_iterator(imagedir, ext=".png", preload=True):
    imfiles = glob.glob(osp.join(imagedir, "*{}".format(ext)))

    data_list = []
    for imfile in sorted(imfiles)[::STRIDE]:
        image = torch.from_numpy(cv2.imread(imfile)).permute(2,0,1)
        intrinsics = torch.as_tensor([fx, fy, cx, cy])
        data_list.append((image, intrinsics))

    for (image, intrinsics) in data_list:
        yield image.cuda(), intrinsics.cuda()

image_raw, _ = next(video_iterator(imagedir))
image = 2 * (image_raw[None,None] / 255.0) - 0.5

# image = (2.0 * (image_raw / 255.0) - 0.5).to(torch.float32).contiguous()
ppi = torch.tensor(PATCHES_PER_IMAGE, dtype=torch.int64, device="cuda")

with torch.no_grad():
    pt_ref = model.patchify(image, patches_per_image=PATCHES_PER_IMAGE, centroid_sel_strat='RANDOM',return_color=True)
    print(f'pt_ref shape: {len(pt_ref)}')
onnx_fmap = torch.empty_like(pt_ref[0])
onnx_gmap = torch.empty_like(pt_ref[1])
onnx_imap = torch.empty_like(pt_ref[2])
onnx_patches = torch.empty_like(pt_ref[3])
onnx_index = torch.empty_like(pt_ref[4])
onnx_clr = torch.empty_like(pt_ref[5])

patchify_io = module_sessions["patchify"].io_binding()
bind_torch_inputs(patchify_io, {"images": image, "patches_per_image": ppi})
for name, t in [
    ("fmap", onnx_fmap),
    ("gmap", onnx_gmap),
    ("imap", onnx_imap),
    ("patches", onnx_patches),
    ("index", onnx_index),
    ("clr", onnx_clr),
]:
    bind_torch_output(patchify_io, name, t)

pytorch_results = []
onnx_results = []
iterations = 10
max_abs = 0.0
all_ok = True

for i in range(iterations):
    if DETERMINISTIC_TEST_MODE:
        # Reset to a known RNG state each iteration for repeatability checks.
        set_all_seeds(seed + i, deterministic=True)

    module_sessions["patchify"].run_with_iobinding(patchify_io)
    onnx_results.append(
        (onnx_fmap.clone(), onnx_gmap.clone(), onnx_imap.clone(), onnx_patches.clone(), onnx_index.clone(), onnx_clr.clone())
    )

    if DETERMINISTIC_TEST_MODE:
        # Use the same RNG state for PyTorch patch selection in this iteration.
        set_all_seeds(seed + i, deterministic=True)

    with torch.no_grad():
        f_pt, g_pt, i_pt, p_pt, idx_pt, c_pt = model.patchify(image, patches_per_image=PATCHES_PER_IMAGE, centroid_sel_strat='RANDOM', return_color=True)
    pytorch_results.append((f_pt, g_pt, i_pt, p_pt, idx_pt, c_pt))

    for sub, a, b in [
        ("patchify/fmap", onnx_fmap, f_pt),
        ("patchify/gmap", onnx_gmap, g_pt),
        ("patchify/imap", onnx_imap, i_pt),
        ("patchify/patches", onnx_patches, p_pt),
        ("patchify/clr", onnx_clr, c_pt),
    ]:
        d, ok = assert_close(sub, a, b)
        max_abs = max(max_abs, d)
        all_ok = all_ok and ok
    idx_equal = torch.equal(onnx_index, idx_pt)
    print(f"patchify/index equal: {idx_equal}")
    all_ok = all_ok and idx_equal
    if i == iterations - 1:
        print(f"Onnx fmap slice: {onnx_fmap.flatten()[:6]}")
        print(f"Pytorch fmap slice: {f_pt.flatten()[:6]}")

output_names = ["fmap", "gmap","imap","patches","index","clr"]
for j in range(6):
    print(f'For ONNX iterations comparison {output_names[j]}')
    for i, t in enumerate(onnx_results[1:]):
        print(f'Result {i}: {torch.equal(onnx_results[0][j], t[j])}')

for j in range(6):
    print(f'For PYTORCH iterations comparison {output_names[j]}')
    for i, t in enumerate(pytorch_results[1:]):
        print(f'Result {i}: {torch.equal(pytorch_results[0][j], t[j])}')

# all_onnx_equal = [torch.equal(onnx_results[0][j], t[j]) for j in range(6) for t in onnx_results[1:]]
# all_pytorch_equal = [torch.equal(pytorch_results[0][j], t[j]) for j in range(6) for t in pytorch_results[1:]]
# print(f"All Onnx Results The Same? {all_onnx_equal}")
# print(f"All Pytorch Results The Same? {all_pytorch_equal}")
record_comparison_summary("patchify", max_abs, all_ok)

pt_ref shape: 6
patchify/fmap: max_abs_diff=2.069e+00  allclose=False
patchify/gmap: max_abs_diff=1.463e+00  allclose=False
patchify/imap: max_abs_diff=1.953e+00  allclose=False
patchify/patches: max_abs_diff=1.530e+02  allclose=False
patchify/clr: max_abs_diff=1.796e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=2.069e+00  allclose=False
patchify/gmap: max_abs_diff=1.451e+00  allclose=False
patchify/imap: max_abs_diff=2.942e+00  allclose=False
patchify/patches: max_abs_diff=1.500e+02  allclose=False
patchify/clr: max_abs_diff=1.820e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=2.069e+00  allclose=False
patchify/gmap: max_abs_diff=1.508e+00  allclose=False
patchify/imap: max_abs_diff=1.758e+00  allclose=False
patchify/patches: max_abs_diff=1.390e+02  allclose=False
patchify/clr: max_abs_diff=1.835e+00  allclose=False
patchify/index equal: True
patchify/fmap: max_abs_diff=2.069e+00  allclose=False
patchify/gmap: max_abs_diff=1.435

In [7]:
#--------------------update (full graph)----------------------
net_u = net.to(torch.float32, copy=False).contiguous()
ctx_u = ctx.to(torch.float32, copy=False).contiguous()
corr_u = corr.to(torch.float32, copy=False).contiguous()
flow_u = torch.zeros(B, E_real, 2, device=net.device, dtype=torch.float32)
ii_u = ii.contiguous()
jj_u = jj.contiguous()
kk_u = kk.contiguous()

with torch.no_grad():
    net_pt, (delta_pt, weight_pt, _) = model.update(net_u, ctx_u, corr_u, flow_u, ii_u, jj_u, kk_u)

net_onnx = torch.empty_like(net_pt)
delta_onnx = torch.empty_like(delta_pt)
weight_onnx = torch.empty_like(weight_pt)

update_io = module_sessions["update"].io_binding()
bind_torch_inputs(
    update_io,
    {"net_in": net_u, "inp": ctx_u, "corr": corr_u, "ii": ii_u, "jj": jj_u, "kk": kk_u},
)
bind_torch_output(update_io, "net_out", net_onnx)
bind_torch_output(update_io, "delta_out", delta_onnx)
bind_torch_output(update_io, "weight_out", weight_onnx)

pytorch_results = []
onnx_results = []
iterations = 10
max_abs = 0.0
all_ok = True

for i in range(iterations):
    module_sessions["update"].run_with_iobinding(update_io)
    onnx_results.append((net_onnx.clone(), delta_onnx.clone(), weight_onnx.clone()))
    with torch.no_grad():
        n_pt, (d_pt, w_pt, _) = model.update(net_u, ctx_u, corr_u, flow_u, ii_u, jj_u, kk_u)
    pytorch_results.append((n_pt, d_pt, w_pt))

    for sub, a, b in [
        ("update/net_out", net_onnx, n_pt),
        ("update/delta_out", delta_onnx, d_pt),
        ("update/weight_out", weight_onnx, w_pt),
    ]:
        d, ok = assert_close(sub, a, b)
        max_abs = max(max_abs, d)
        all_ok = all_ok and ok
    if i == iterations - 1:
        print(f"Onnx net_out slice: {net_onnx.flatten()[:6]}")
        print(f"Pytorch net_out slice: {n_pt.flatten()[:6]}")

all_onnx_equal = all(
    all(torch.equal(onnx_results[0][j], t[j]) for j in range(3)) for t in onnx_results[1:]
)
all_pytorch_equal = all(
    all(torch.equal(pytorch_results[0][j], t[j]) for j in range(3)) for t in pytorch_results[1:]
)
print(f"All Onnx Results The Same? {all_onnx_equal}")
print(f"All Pytorch Results The Same? {all_pytorch_equal}")
record_comparison_summary("update", max_abs, all_ok)

update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e-02  allclose=False
update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e-02  allclose=False
update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e-02  allclose=False
update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e-02  allclose=False
update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e-02  allclose=False
update/net_out: max_abs_diff=8.385e+00  allclose=False
update/delta_out: max_abs_diff=1.099e+01  allclose=False
update/weight_out: max_abs_diff=6.807e

2026-04-07 17:08:44.245372047 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2209,384} for output /agg_kk/Identity_3_output_0
2026-04-07 17:08:44.249414989 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,450,384} for output /agg_ij/Identity_3_output_0
2026-04-07 17:08:44.252416751 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2209,384} for output /agg_kk/Identity_3_output_0
2026-04-07 17:08:44.252541701 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,450,384} for output /agg_ij/Identity_3_output_0
2026-04-07 17:08:44.254824905 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2209,384} for output 

In [8]:
#--------------------corr----------------------
corr_t = corr.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
corr_onnx_out = torch.empty((B, E_real, D), device=corr_t.device, dtype=torch.float32)
corr_io_binding = module_sessions['corr'].io_binding()
bind_torch_inputs(corr_io_binding, {'corr_input': corr_t})

corr_io_binding.bind_output(
    name='corr_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(corr_onnx_out.shape),
    buffer_ptr=corr_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['corr'].run_with_iobinding(corr_io_binding)
    onnx_results.append(corr_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        corr_pytorch_out = model.update.corr(corr_t)
    pytorch_results.append(corr_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("corr", corr_onnx_out, corr_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {corr_onnx_out}')
        print(f'Pytorch Output: {corr_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("corr", max_abs, all_ok)


corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
corr: max_abs_diff=1.819e-03  allclose=False
Onnx Output: tensor([[[ 1.8998265266, -3.6251235008, -0.1360934377,  ..., -0.9074153304,  1.4647865295,  4.4469285011],
         [-1.1548005342, -1.0765366554,  2.1619408131,  ...,  0.6352051497, -2.0529170036,  0.1112378836],
         [-1.6618080139, -0.8971620202,  1.9045437574,  ...,  0.6823101044,  0.0132831819, -0.0790750980],
         ...,
         [-0.0082055731, -1.8365343809, -1.4504175186,  ...,  0.1222935840, -0.9207568169,  2.0202248096],
         [-0.7767445445, -0.7698490024, -0.5215212107,  ...,  1.3732770681,  0.2577459514,  3.4

In [9]:
#--------------------norm----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
norm_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
norm_io_binding = module_sessions['norm'].io_binding()
bind_torch_inputs(norm_io_binding, {'net_input': net_t})

norm_io_binding.bind_output(
    name='norm_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(norm_onnx_out.shape),
    buffer_ptr=norm_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['norm'].run_with_iobinding(norm_io_binding)
    onnx_results.append(norm_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        norm_pytorch_out = model.update.norm(net_t)
    pytorch_results.append(norm_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("norm", norm_onnx_out, norm_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {norm_onnx_out}')
        print(f'Pytorch Output: {norm_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("norm", max_abs, all_ok)


norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
norm: max_abs_diff=0.000e+00  allclose=True
Onnx Output: tensor([[[-0.0561140664,  0.0158388298,  0.0219458174,  ..., -0.0596615039, -0.0202397741, -0.0293612294],
         [-0.0561140664,  0.0158388298,  0.0219458174,  ..., -0.0596615039, -0.0202397741, -0.0293612294],
         [-0.0561140664,  0.0158388298,  0.0219458174,  ..., -0.0596615039, -0.0202397741, -0.0293612294],
         ...,
         [-0.0561140664,  0.0158388298,  0.0219458174,  ..., -0.0596615039, -0.0202397741, -0.0293612294],
         [-0.0561140664,  0.0158388298,  0.0219458174,  ..., -0.0596615039, -0.0202397741, -0.0293612294]

In [10]:
#--------------------c1----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
c1_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c1_io_binding = module_sessions['c1'].io_binding()
bind_torch_inputs(c1_io_binding, {'c1_input': net_t})

c1_io_binding.bind_output(
    name='c1_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c1_onnx_out.shape),
    buffer_ptr=c1_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['c1'].run_with_iobinding(c1_io_binding)
    onnx_results.append(c1_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        c1_pytorch_out = model.update.c1(net_t)
    pytorch_results.append(c1_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("c1", c1_onnx_out, c1_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {c1_onnx_out}')
        print(f'Pytorch Output: {c1_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("c1", max_abs, all_ok)


c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
c1: max_abs_diff=2.354e-05  allclose=False
Onnx Output: tensor([[[ 0.0634784400, -0.0417013466, -0.0909733102,  ..., -0.1075124592, -0.0762622580, -0.0080392389],
         [ 0.0634784400, -0.0417013466, -0.0909733102,  ..., -0.1075124592, -0.0762622580, -0.0080392389],
         [ 0.0634784400, -0.0417013466, -0.0909733102,  ..., -0.1075124592, -0.0762622580, -0.0080392389],
         ...,
         [ 0.0634784400, -0.0417013466, -0.0909733102,  ..., -0.1075124592, -0.0762622580, -0.0080392389],
         [ 0.0634784400, -0.0417013466, -0.0909733102,  ..., -0.1075124592, -0.0762622580, -0.0080392389],
        

In [11]:
#--------------------c2----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
c2_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
c2_io_binding = module_sessions['c2'].io_binding()
bind_torch_inputs(c2_io_binding, {'c2_input': net_t})

c2_io_binding.bind_output(
    name='c2_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(c2_onnx_out.shape),
    buffer_ptr=c2_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['c2'].run_with_iobinding(c2_io_binding)
    onnx_results.append(c2_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        c2_pytorch_out = model.update.c2(net_t)
    pytorch_results.append(c2_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("c2", c2_onnx_out, c2_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {c2_onnx_out}')
        print(f'Pytorch Output: {c2_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("c2", max_abs, all_ok)


c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
c2: max_abs_diff=4.572e-05  allclose=False
Onnx Output: tensor([[[-0.0084335320, -0.0410595387, -0.0501707904,  ..., -0.1780614406,  0.0488621294,  0.0165268593],
         [-0.0084335320, -0.0410595387, -0.0501707904,  ..., -0.1780614406,  0.0488621294,  0.0165268593],
         [-0.0084335320, -0.0410595387, -0.0501707904,  ..., -0.1780614406,  0.0488621294,  0.0165268593],
         ...,
         [-0.0084335320, -0.0410595387, -0.0501707904,  ..., -0.1780614406,  0.0488621294,  0.0165268593],
         [-0.0084335320, -0.0410595387, -0.0501707904,  ..., -0.1780614406,  0.0488621294,  0.0165268593],
        

In [12]:
#--------------------agg_kk----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
_, jx_kk = torch.unique(kk, return_inverse=True)

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
agg_kk_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_kk_io_binding = module_sessions['agg_kk'].io_binding()
bind_torch_inputs(agg_kk_io_binding, {'agg_kk_input': net_t, 'jx_input': jx_kk})

agg_kk_io_binding.bind_output(
    name='agg_kk_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_kk_onnx_out.shape),
    buffer_ptr=agg_kk_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['agg_kk'].run_with_iobinding(agg_kk_io_binding)
    onnx_results.append(agg_kk_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        agg_kk_pytorch_out = model.update.agg_kk(net_t, jx_kk)
    pytorch_results.append(agg_kk_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("agg_kk", agg_kk_onnx_out, agg_kk_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {agg_kk_onnx_out}')
        print(f'Pytorch Output: {agg_kk_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("agg_kk", max_abs, all_ok)


agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
agg_kk: max_abs_diff=3.563e-05  allclose=False
Onnx Output: tensor([[[-0.0817982852,  0.0156192100,  0.0941733196,  ..., -0.1102893725, -0.0883082598, -0.0389204472],
         [-0.0817982852,  0.0156192100,  0.0941733196,  ..., -0.1102893725, -0.0883082598, -0.0389204472],
         [-0.0817982852,  0.0156192100,  0.0941733196,  ..., -0.1102893725, -0.0883082598, -0.0389204472],
         ...,
         [-0.0817982852,  0.0156192100,  0.0941733196,  ..., -0.1102893725, -0.0883082598, -0.0389204472],
         [-0.0817982852,  0.0156192100,  0.0941733196,  ..., -0.1102893725,

2026-04-07 17:08:44.411628144 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-07 17:08:44.413164540 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-07 17:08:44.414044849 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-07 17:08:44.414840624 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-04-07 17:08:44.415775410 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,2208,384} for output /Identity_3_output_0
2026-

In [13]:
#--------------------agg_ij----------------------

net_t = net.to(torch.float32, copy=False).contiguous()
iijj = ii * 12345 + jj
_, jx_ij = torch.unique(iijj, return_inverse=True)

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
agg_ij_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
agg_ij_io_binding = module_sessions['agg_ij'].io_binding()
bind_torch_inputs(agg_ij_io_binding, {'agg_ij_input': net_t, 'jx_input': jx_ij})

agg_ij_io_binding.bind_output(
    name='agg_ij_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(agg_ij_onnx_out.shape),
    buffer_ptr=agg_ij_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['agg_ij'].run_with_iobinding(agg_ij_io_binding)
    onnx_results.append(agg_ij_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        agg_ij_pytorch_out = model.update.agg_ij(net_t, jx_ij)
    pytorch_results.append(agg_ij_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("agg_ij", agg_ij_onnx_out, agg_ij_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {agg_ij_onnx_out}')
        print(f'Pytorch Output: {agg_ij_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("agg_ij", max_abs, all_ok)


2026-04-07 17:08:44.439548040 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0
2026-04-07 17:08:44.440995334 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0
2026-04-07 17:08:44.441736519 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0
2026-04-07 17:08:44.442388479 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0
2026-04-07 17:08:44.443045041 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0
2026-04-07

agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
agg_ij: max_abs_diff=2.746e-05  allclose=False
Onnx Output: tensor([[[ 0.0172811300, -0.0049406830, -0.0565118752,  ..., -0.1261858940,  0.1167213321, -0.0615061261],
         [ 0.0172811300, -0.0049406830, -0.0565118752,  ..., -0.1261858940,  0.1167213321, -0.0615061261],
         [ 0.0172811300, -0.0049406830, -0.0565118752,  ..., -0.1261858940,  0.1167213321, -0.0615061261],
         ...,
         [ 0.0172811300, -0.0049406830, -0.0565118752,  ..., -0.1261858940,  0.1167213321, -0.0615061261],
         [ 0.0172811300, -0.0049406830, -0.0565118752,  ..., -0.1261858940,

2026-04-07 17:08:44.446325158 [W:onnxruntime:, execution_frame.cc:870 VerifyOutputSizes] Expected shape from model of {-1} does not match actual shape of {1,449,384} for output /Identity_3_output_0


In [14]:
#--------------------gru----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
gru_onnx_out = torch.empty((B, E_real, D), device=net_t.device, dtype=torch.float32)
gru_io_binding = module_sessions['gru'].io_binding()
bind_torch_inputs(gru_io_binding, {'gru_input': net_t})

gru_io_binding.bind_output(
    name='net_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(gru_onnx_out.shape),
    buffer_ptr=gru_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['gru'].run_with_iobinding(gru_io_binding)
    onnx_results.append(gru_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        gru_pytorch_out = model.update.gru(net_t)
    pytorch_results.append(gru_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("gru", gru_onnx_out, gru_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {gru_onnx_out}')
        print(f'Pytorch Output: {gru_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("gru", max_abs, all_ok)


gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
gru: max_abs_diff=6.900e-04  allclose=False
Onnx Output: tensor([[[-0.8305189610,  0.2820754647,  0.6968061924,  ..., -0.4081175923, -0.4293887615, -0.2446229607],
         [-0.8305189610,  0.2820754647,  0.6968061924,  ..., -0.4081175923, -0.4293887615, -0.2446229607],
         [-0.8305189610,  0.2820754647,  0.6968061924,  ..., -0.4081175923, -0.4293887615, -0.2446229607],
         ...,
         [-0.8305189610,  0.2820754647,  0.6968061924,  ..., -0.4081175923, -0.4293887615, -0.2446229607],
         [-0.8305189610,  0.2820754647,  0.6968061924,  ..., -0.4081175923, -0.4293887615, -0.2446229607]

In [15]:
#--------------------w----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
w_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
w_io_binding = module_sessions['w'].io_binding()
bind_torch_inputs(w_io_binding, {'net_input': net_t})

w_io_binding.bind_output(
    name='weight_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(w_onnx_out.shape),
    buffer_ptr=w_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['w'].run_with_iobinding(w_io_binding)
    onnx_results.append(w_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        w_pytorch_out = model.update.w(net_t)
    pytorch_results.append(w_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("w", w_onnx_out, w_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {w_onnx_out}')
        print(f'Pytorch Output: {w_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("w", max_abs, all_ok)


w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
w: max_abs_diff=2.980e-08  allclose=True
Onnx Output: tensor([[[0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.4960961938],
         [0.4820420146, 0.49609

In [16]:
#--------------------d----------------------

net_t = net.to(torch.float32, copy=False).contiguous()

pytorch_results = []
onnx_results = []

# -----------ONNX----------------------
d_onnx_out = torch.empty((B, E_real, 2), device=net_t.device, dtype=torch.float32)
d_io_binding = module_sessions['d'].io_binding()
bind_torch_inputs(d_io_binding, {'net_input': net_t})

d_io_binding.bind_output(
    name='delta_out',
    device_type="cuda",
    device_id=0,
    element_type=np.float32,
    shape=tuple(d_onnx_out.shape),
    buffer_ptr=d_onnx_out.data_ptr(),
)
iterations = 10
max_abs = 0.0
all_ok = True
for i in range(iterations):
    # ----------onnx ------------------------
    module_sessions['d'].run_with_iobinding(d_io_binding)
    onnx_results.append(d_onnx_out)
    # -----------Pytorch----------------------
    with torch.no_grad():
        d_pytorch_out = model.update.d(net_t)
    pytorch_results.append(d_pytorch_out)
    # -----------Comparison---------------------
    d, ok = assert_close("d", d_onnx_out, d_pytorch_out)
    max_abs = max(max_abs, d)
    all_ok = all_ok and ok
    if i == iterations - 1:
        print(f'Onnx Output: {d_onnx_out}')
        print(f'Pytorch Output: {d_pytorch_out}')

all_onnx_equal = all(torch.equal(onnx_results[0], t) for t in onnx_results[1:])
all_pytorch_equal = all(torch.equal(pytorch_results[0], t) for t in pytorch_results[1:])

print(f'All Onnx Results The Same? {all_onnx_equal}')
print(f'All Pytorch Results The Same? {all_pytorch_equal}')
record_comparison_summary("d", max_abs, all_ok)


d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
d: max_abs_diff=0.000e+00  allclose=True
Onnx Output: tensor([[[ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
         [ 0.0442100912, -0.0233600959],
   

In [17]:
# Summary: max |ONNX − PyTorch| over all repeat iterations (patchify, update, then modular blocks)
comparison_df = pd.DataFrame(comparison_rows, index=None)
comparison_df
#input_payload

,model,max_abs_diff,allclose
0,patchify,1.540000e+02,False
1,update,1.098939e+01,False
2,corr,1.818657e-03,False
3,norm,0.000000e+00,True
4,c1,2.354383e-05,False
5,c2,4.571676e-05,False
6,agg_kk,3.562868e-05,False
7,agg_ij,2.746284e-05,False
8,gru,6.899834e-04,False
9,w,2.980232e-08,True
